In [154]:
#Dependencias para usar 
from selenium import webdriver
import time
import os
import time
import requests
from bs4 import BeautifulSoup
from pprint import pprint
import pandas as pd
import json
from lxml import html
import re
import csv
import numpy as np
from selenium import webdriver
from selenium.webdriver.chrome.service import Service #para genrar un objeto de tipo Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.remote.webelement import WebElement
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    NoSuchElementException,
    TimeoutException,
    WebDriverException,
    )
from webdriver_manager.chrome import ChromeDriverManager #descargar lo
import pandas as pd

In [160]:
options = Options()
options.add_argument('--ignore-certificate-errors')
options.add_argument('--disable-notifications')


driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()),options=options)
driver.get("https://www.google.com/")
#primera busqueda
search = driver.find_element(By.NAME, value="q")
search.send_keys("Restaurantes CDMX")
search.send_keys(Keys.ENTER)

time.sleep(5)
avanzar = driver.find_elements(By.LINK_TEXT,'Más lugares')
for item in avanzar:
  item.click()

In [161]:
def res_hasta_pag_n (n,text):
    resultados = []
    for i in range(1,n+1):
      otr = driver.find_elements(By.LINK_TEXT,'{}'.format(str(i)))
      for item in otr:
        item.click()
        time.sleep(5)
      nueva_pagina = [contenido.text for contenido in driver.find_elements(By.CLASS_NAME,text)]
      resultados.extend(nueva_pagina)
    print("se terminó la busqueda")
    return resultados

In [162]:
resultados_busqueda = res_hasta_pag_n(10,"rllt__details")

se terminó la busqueda


In [163]:
print(resultados_busqueda)

['La Bikina México\nAnuncio·4.2\n(181) · Restaurante mexicano\nCiudad de México, CDMX\nAbierto ⋅ Cierra a la 01:00\nEntrega a domicilio', 'Cambalache Oasis Coyoacán\n4.6\n(2.6 K) · $$$ · Argentina\nAvenida Miguel Ángel de Quevedo 227 Local R-03-04\nConsumo en el lugar·\nPara llevar·\nEntrega sin contacto', 'San Ángel Inn\n4.6\n(8.1 K) · $$$ · Mexicana\nDiego Rivera 50\nConsumo en el lugar·\nPara llevar·\nNo ofrece servicio de entrega a domicilio', 'Saks San Ángel\n4.6\n(4.4 K) · $$$ · Mexicana\nPl. San Jacinto 9\nConsumo en el lugar·\nPara llevar·\nEntrega sin contacto', 'Sonora Grill\n4.5\n(6.7 K) · $$$ · Parrilla\nAv. Coyoacán 1955\nConsumo en el lugar·\nRetiros en la puerta·\nEntrega sin contacto', 'Parrilla Urbana\n4.2\n(1.9 K) · $$ · Parrilla\nAv. División del Nte. 911\nConsumo en el lugar·\nPara llevar·\nEntrega sin contacto', 'Daikoku\n4.4\n(4 K) · $$ · Japonesa\nLondres 348\nCierra pronto: ⋅ 23:30\nEntrega: Finaliza pronto ⋅ 23:00', 'La Bipo\n4.2\n(5.6 K) · $$ · Mexicana\nMalin

In [148]:
print(resultados_busqueda[0])

['Te Mataré Santana\nAnuncio·4.3\n(300) · $$ · Restaurante\nCiudad de México, CDMX\nAbierto ⋅ Cierra a las 00:00\nEntrega a domicilio', 'ASADOR LIBANÉS\n4.5\n(955) · $$ · Restaurante libanés\nBenito Juárez, CDMX\nAbierto ⋅ Cierra a las 00:00\nRetiros en la puerta·\nEntrega sin contacto', 'Cambalache Oasis Coyoacán\n4.6\n(2.6 K) · $$$ · Argentina\nAvenida Miguel Ángel de Quevedo 227 Local R-03-04\n"El mejor restaurante de comida argentina en Mexico."', 'Los Danzantes Coyoacán\n4.4\n(3.6 K) · $$$ · Mexicana\nParque Centenario 12\nCierra pronto: ⋅ 23:00\n"De mis lugares favoritos para comer comida mexicana."', 'San Ángel Inn\n4.6\n(8.1 K) · $$$ · Mexicana\nDiego Rivera 50\n"Restaurante tradicional de comida mexicana en una exhacienda"', 'Mesón Antigua Santa Catarina\n4.3\n(2.2 K) · $$ · Mexicana\nJardín Sta Catarina 6\nCierra pronto: ⋅ 23:00\n"Restaurante de comida mexicana."', 'Sonora Grill\n4.5\n(6.7 K) · $$$ · Parrilla\nAv. Coyoacán 1955\n"Alimentos de calidad y sabor como en todos sus

In [159]:
resultados_busqueda = [cadena.split(" ⋅ ") for cadena in resultados_busqueda]
print(resultados_busqueda)


[['La Bikina México\nAnuncio·4.2\n(181) · Restaurante mexicano\nCiudad de México, CDMX\nAbierto', 'Cierra a la 01:00\nEntrega a domicilio'], ['Te Mataré Santana\nAnuncio·4.3\n(300) · $$ · Restaurante\nCiudad de México, CDMX\nAbierto', 'Cierra a las 00:00\nEntrega a domicilio'], ['Cambalache Oasis Coyoacán\n4.6\n(2.6 K) · $$$ · Argentina\nAvenida Miguel Ángel de Quevedo 227 Local R-03-04\n"El mejor restaurante de comida argentina en Mexico."'], ['San Ángel Inn\n4.6\n(8.1 K) · $$$ · Mexicana\nDiego Rivera 50\n"Menú poco convencional de comida mexicana."'], ['Sonora Grill\n4.5\n(6.7 K) · $$$ · Parrilla\nAv. Coyoacán 1955\n"Toda la comida muy sabrosa"'], ['Porfirios Coapa | Restaurante de comida mexicana\n4.5\n(1.8 K) · Mexicana\nCDMX\nConsumo en el lugar·\nRetiros en la puerta·\nEntrega sin contacto'], ['Restaurante El Cardenal\n4.7\n(10 k) · $$$ · Mexicana\nAv. de la Paz 32\nCerrado', 'Abre a las 08:00 del jue\n"comida mexicana muy bien preparada y rica"'], ['La Bipo\n4.2\n(5.6 K) · $$ ·